In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import uuid
from pyspark.sql.window import Window


In [0]:
spark.sql("""
          create table if not exists datatocrunch_novacart.silver_schema.ingestion_control(
              medallion_layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp 
          )
          
          """)

# Step 3 - Helper functions

This cell contains reusable logic for Silver:

- upsert_to_silver() merges cleaned / transformed rows into the Silver target table
- get_last_processed_bronze_ingested_at() reads the Silver watermark
- upsert_silver_control() updates the Silver control table
- get_incremental_bronze() reads only new Bronze rows that Silver has not processed yet

In [0]:
def upsert_to_silver(source_df,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        target=DeltaTable.forName(spark,target_table)
        (
            target.alias("t")
            .merge(source_df.alias("s"),f"t.{join_key}=s.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.format("delta").saveAsTable(target_table)
            


In [0]:
from pyspark.sql import functions as F

def get_last_processed_bronze_ingested_at(entity_name: str):
    row = (
        spark.table("datatocrunch_novacart.silver_schema.ingestion_control")
        .filter(
            (F.col("medallion_layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("last_processed_bronze_ingested_at").desc())
        .limit(1)
        .first()
    )

    if row is None:
        return None
    else:
        return row["last_processed_bronze_ingested_at"]

In [0]:

def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    source_df = spark.createDataFrame([{
        "medallion_layer": "silver",
        "entity_name": entity_name,
        "last_processed_bronze_run_id": last_processed_bronze_run_id,
        "last_processed_bronze_ingested_at": last_processed_bronze_ingested_at,
        "rows_merged": rows_merged,
        "run_status": "success",
        "silver_run_id": str(uuid.uuid4()),
        "updated_at": datetime.now()
    }])

    target = DeltaTable.forName(spark,"datatocrunch_novacart.silver_schema.ingestion_control")
    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.medallion_layer = s.medallion_layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id":"s.last_processed_bronze_run_id",
                "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
                "rows_merged": "s.rows_merged",
                "run_status": "s.run_status",
                "silver_run_id": "s.silver_run_id",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_processed_entity_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df = spark.read.table(bronze_table)

    if last_processed_entity_ingested_at is None:
        return bronze_df, None

    return (
        bronze_df.filter(
            F.col("bronze_ingested_at") > F.lit(last_processed_entity_ingested_at)
        ),
        last_processed_entity_ingested_at
    )

#Orders incremental processing

In [0]:

spark.sql("use catalog datatocrunch_novacart")
orders_inc_load,last_orders_ingested_at=get_incremental_bronze("datatocrunch_novacart.bronze_schema.orders_raw","orders")

#new inc load count
orders_inc_load_count=orders_inc_load.count()
print(f"Order table has {orders_inc_load_count} new records to process from bronze to silver")
silver_run_id=str(uuid.uuid4())
if orders_inc_load_count>0:
    #create  a window which shows new order record for each order_id
    order_window=Window.partitionBy("order_id").orderBy(
        F.col("updated_at").desc(),
        F.col("bronze_ingested_at").desc()
    )

    #orders_cleaned_data
    orders_cleaned=(orders_inc_load.withColumn("order_status",F.upper(F.trim("order_status")))
    .withColumn("order_status",F.when(F.col("order_status")=="",F.lit("None")).otherwise(F.col("order_status")))
    .withColumn("order_amount",
    F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
    .withColumn("order_amount",F.when(
                F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""),None).otherwise(F.col("order_amount"))
    )
    .withColumn("order_amount",F.col("order_amount").cast("double"))
    .withColumn("created_at",F.to_timestamp("created_at"))
    .withColumn("updated_at",F.to_timestamp("updated_at"))
    .withColumn("row_rank",F.row_number().over(order_window))
    .filter(F.col("row_rank")==1)
    .drop("row_rank")
    .withColumn("silver_run_id",F.lit("silver_run_id"))
    )

    upsert_to_silver(orders_cleaned,"datatocrunch_novacart.silver_schema.orders_cleaned","order_id")

    orders_validated=(
    orders_cleaned.withColumn("to_be_verified_by_orders_team",
    F.when(F.col("customer_id").isNull(),"verify_customer_id")
    .when(F.col("product_id").isNull(),"verify_product_id")
    .when((F.col("order_status").isNull()) | (F.trim(F.col("order_status"))==""),"verify_order_status")
    .when((F.col("order_amount").isNull()) | (F.col("order_amount")<=0),"verify_order_amount")
    .otherwise("No issues"))
    .withColumn(
        "check_order_amount",
        F.when(
            (F.col("order_amount").isNull()) |
            (F.col("order_amount") <= 0),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn("order_date", F.to_date("created_at"))
    .withColumn("order_month", F.month("created_at"))
    .withColumn("order_year", F.year("created_at"))
    .withColumn("order_day", F.dayofmonth("created_at"))
    .withColumn("order_dow", F.date_format("created_at", "E"))
)

                
    orders_good=orders_validated.filter(F.col("to_be_verified_by_orders_team")=="No issues")
    orders_bad=orders_validated.filter(F.col("to_be_verified_by_orders_team")!="No issues").withColumn("orders_quarantine",F.current_timestamp())

    upsert_to_silver(orders_good,"datatocrunch_novacart.silver_schema.orders_transformed","order_id")

    orders_bad.write.format("delta").mode("append").saveAsTable("datatocrunch_novacart.silver_schema.orders_quarantine")

    mx_ingested = (
    orders_inc_load
    .agg(
        F.max("bronze_ingested_at").alias("mx_ts")
    )
    .first()["mx_ts"])
                                     
    mx_run_id = (orders_inc_load
    .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
    .select("bronze_runid")
    .first()["bronze_runid"])

    upsert_silver_control("orders",mx_run_id,mx_ingested,orders_good.count())
else:
    print("No new records from orders table to load from bronze to silver")

                                                


#Products incremental processing

In [0]:
products_inc_load,last_products_ingested_at=get_incremental_bronze("datatocrunch_novacart.bronze_schema.products_raw","products")
silver_run_id=str(uuid.uuid4())
#new inc load count
products_inc_load_count=products_inc_load.count()
print(f"Products table has {products_inc_load_count} new records to process from bronze to silver")

if products_inc_load_count>0:
    products_window=Window.partitionBy("product_id").orderBy(F.col("updated_at").desc(),F.col("bronze_ingested_at").desc())

    products_cleaned=(
        products_inc_load.withColumn("product_name",F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name",F.when(F.col("product_name")=="",F.lit(None)).otherwise(F.col("product_name")))
        .withColumn("category",F.upper("category"))
        .withColumn("product_code",F.upper("product_name"))
        .withColumn("product_name",F.regexp_replace(F.col("product_name"),r"[-]"," "))
        .withColumn("product_code",F.regexp_replace(F.col("product_name"),r"(?i)^prod_","PRODUCT "))
        .withColumn("category",F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"),"ELECTRONICS").otherwise(F.col("category")))
        .withColumn("price",F.trim(F.col("price")))
        .withColumn("price",F.regexp_replace(F.col("price"),r"[$]",""))
        .withColumn("price",F.regexp_replace(F.col("price"),r"[,]","."))
        .withColumn("price",F.expr("try_cast(price as double)"))
        .withColumn("updated_at",F.to_timestamp("updated_at"))
        .withColumn("row_rank",F.row_number().over(products_window)).filter(F.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run_id))
    )
    
    upsert_to_silver(products_cleaned,"datatocrunch_novacart.silver_schema.products_cleaned","product_id")

    products_validated=(

        products_cleaned.withColumn("to_be_verified_by_products_team",F.when(F.col("product_name").isNull(),"verify_product_name")
        .when(F.col("category").isNull(),"verify_category")
        .when((F.col("price").isNull()) | (F.col("price") <= 0),"verify_price")

        .otherwise("No issues"))
        .withColumn("check_product_price",F.when((F.col("price").isNull()) | (F.col("price") <= 0),"invalid_price")
        .otherwise("valid_price"))
        
    )

    products_good = products_validated.filter(
    (F.col("to_be_verified_by_products_team") == "No issues") &
    (F.col("check_product_price") == "valid_price")
)
    products_bad = (
    products_validated
    .filter(
        (F.col("to_be_verified_by_products_team") != "No issues") |
        (F.col("check_product_price") != "valid_price")
    )
    .withColumn("products_quarantine", F.current_timestamp())
    )

    upsert_to_silver(products_good,"datatocrunch_novacart.silver_schema.products_transformed","product_id")

    products_bad.write.format("delta").mode("append").saveAsTable("datatocrunch_novacart.silver_schema.products_quarantine")

    mx_ingested = (
    products_inc_load
    .agg(
        F.max("bronze_ingested_at").alias("mx_ts")
    )
    .first()["mx_ts"])
                                     
    mx_run_id = (products_inc_load
    .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
    .select("bronze_runid")
    .first()["bronze_runid"])

    upsert_silver_control("products",mx_run_id,mx_ingested,products_good.count())
else:
    print("No new records from products table to load from bronze to silver")



In [0]:
%sql
select * from datatocrunch_novacart.silver_schema.products_transformed

In [0]:
payments_inc_load,last_payments_ingested_at=get_incremental_bronze("datatocrunch_novacart.bronze_schema.payments_raw","payments")
silver_run_id=str(uuid.uuid4())
#new inc load count
payments_inc_load_count=payments_inc_load.count()
print(f"payments table has {payments_inc_load_count} new records to process from bronze to silver")

if payments_inc_load_count>0:
    payments_window=Window.partitionBy("payment_id").orderBy(F.col("processed_at").desc(),F.col("bronze_ingested_at").desc())
    payments_cleaned=(
        payments_inc_load.withColumn("payment_status",F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status",F.when(F.col("payment_status")=="",F.lit(None)).otherwise(F.col("payment_status")))
        .withColumn("paid_amount",F.trim(F.col("paid_amount")))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r"[$]",""))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r"[,]","."))
        .withColumn("paid_amount",F.expr("try_cast(paid_amount as double)"))
        
        .withColumn("processed_at",F.to_timestamp("processed_at"))
        .withColumn("row_rank",F.row_number().over(payments_window)).filter(F.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run_id))
    )


    upsert_to_silver(payments_cleaned,"datatocrunch_novacart.silver_schema.payments_cleaned","payment_id")

    payments_validated=(

        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",F.when(F.col("payment_id").isNull(),"verify_payment_id")
        .when(F.col("payment_status").isNull(),"verify_payment_status")
        .when((F.col("paid_amount").isNull()) | (F.col("paid_amount") <= 0),"verify_paid_amount")
        .otherwise("No issues"))
        .withColumn("check_paid_amount",F.when((F.col("paid_amount").isNull()) | (F.col("paid_amount") <= 0),F.lit(True)).otherwise(F.lit(False))
    ))

    payments_good=(payments_validated.filter(F.col("to_be_verified_by_payments_team") == "No issues"))

    payments_bad = (
    payments_validated
    .filter(
        (F.col("to_be_verified_by_payments_team") != "No issues") |
        (F.col("check_paid_amount") == True)
    )
    .withColumn("payments_quarantine", F.current_timestamp())
    )

    upsert_to_silver(payments_good,"datatocrunch_novacart.silver_schema.payments_transformed","payment_id")

    payments_bad.write.format("delta").mode("append").saveAsTable("datatocrunch_novacart.silver_schema.payments_quarantine")

    mx_ingested = (
    payments_inc_load
    .agg(
        F.max("bronze_ingested_at").alias("mx_ts")
    )
    .first()["mx_ts"])
                                     
    mx_run_id = (payments_inc_load
    .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
    .select("bronze_runid")
    .first()["bronze_runid"])

    upsert_silver_control("payments",mx_run_id,mx_ingested,payments_good.count())
else:
    print("No new records from payments table to load from bronze to silver")





In [0]:
%sql
select * from datatocrunch_novacart.silver_schema.payments_quarantine

In [0]:
%sql
select * from datatocrunch_novacart.silver_schema.ingestion_control
--delete from datatocrunch_novacart.silver_schema.ingestion_control where entity_name=="payments"